# Example notebook computing forecasts and losses using random/trained weights

The prerequisites of the notebook are:

1. The dataset and normalization artifacts.
2. The global ocean mesh.
3. Trained model

All settings are serialized and saved on disk for later reuse.

In [ ]:
import pathlib
import platform

import cartopy.crs as ccrs
import grain.python as grain
import haiku as hk
import jax
import numpy as np
import panel as pn
from graphcast.ocean_mesh_utils import read_mesh

from graphcast import cli_utils, training_utils as trn_utils, xarray_jax, checkpoint, rollout
from graphcast.dataloader import ARCODataSource
from graphcast.model import TaskConfig, CheckPoint

pn.extension()

In [ ]:
# The notebook is thought to be executed both on local hardware and on Leonardo. When running on laptop it uses a coarser grid and mesh.

if platform.node().endswith('leonardo.local'):
    code_path = pathlib.Path("/leonardo_scratch/fast/OGS23_PRACE_IT_0/scampane/xcast")
    data_path = pathlib.Path("/leonardo_scratch/large/userexternal/scampane/xcast/")
    resolution = "0p25"
    batch_size = 1
else:
    code_path = pathlib.Path("../")
    data_path = pathlib.Path("../")
    train_path = None
    resolution = "1"
    batch_size = 1
    jax.config.update('jax_num_cpu_devices', 4)

In [ ]:
with (data_path / 'data/models/full_v2.ckpt').open('rb') as file:
    ckpt = checkpoint.load(file, CheckPoint)

In [ ]:
def get_dashboard(dataset, targets, title="", batch=0, width=600, height=480, absolute_scale=False, **kwargs):
    """ Display a dashboard showing data with possibly all of (batch, time, level, lat, lon) dimensions."""

    variable_selector = pn.widgets.Select(
        name='Variable',
        options=list(dataset.data_vars))

    if 'level' in dataset.coords:
        level_options = {val: idx for idx, val in enumerate(dataset['level'].to_numpy())}
    else:
        level_options = []

    level_selector = pn.widgets.Select(
        name='Level',
        options=level_options)

    if 'time' in dataset.coords:
        time_options = {val: idx for idx, val in enumerate(dataset['time'].dt.days.to_numpy())}
    else:
        time_options = []
        
    time_selector = pn.widgets.DiscreteSlider(
        name='Day',
        options=time_options)

    def get_display(ds, **kwargs):
    
        @pn.depends(variable_selector.param.value, level_selector.param.value, time_selector.param.value)
        def display(selected_variable, selected_level, selected_time):
            try:
                da = ds[selected_variable]
                if absolute_scale:
                    clim = (da.min(), da.max())
                else:
                    clim=None
                if 'level' in da.dims:
                    da = da.isel(level=selected_level)
                    da = da.drop_vars('level')
                if 'time' in da.dims:
                    da = da.isel(time=selected_time)
                    da = da.drop_vars('time')
                if 'batch' in dataset.dims:
                    da = da.isel(batch=batch)
                return da.hvplot.image("lon", "lat", clim=clim, width=width, height=height, **kwargs)
            except Exception as e:
                # Return an informative message if an error occurs during plotting
                return pn.pane.Markdown(f"### Error generating plot for var={selected_variable}, level={selected_level}, batch={batch}, time={selected_time}: {e}")
        
        return display

    dashboard = pn.WidgetBox(
        pn.Card(
            variable_selector,
            level_selector,
            time_selector),
        pn.Row(
            get_display(dataset, title="Forecast", **kwargs),
            get_display(dataset - targets, title="Error", cmap='bwr', **kwargs)))
    
    return dashboard

In [ ]:
config_path = code_path / "configs/training/launch_full.toml"

configs = cli_utils.Configs.read(config_path)

In [ ]:
mesh_data = trn_utils.get_mesh(data_path, configs)

In [ ]:
mask = trn_utils.get_mask(data_path, configs)
grid_lat = mask['lat'].to_numpy()
grid_lon = mask['lon'].to_numpy()
grid_mask = mask.transpose('lat', 'lon').to_numpy()

mean_by_level, stddev_by_level, diffs_stddev_by_level = trn_utils.get_artifacts(data_path, configs)

In [ ]:
dataset_path = data_path / f"data/dataset/dataset_tres-1d_res-{resolution}_levels-10_arco"
from_date = "2020-06-30"

# replace with function from training utils
task_config = TaskConfig(
    input_variables=configs.get('task.input_variables', required=True),
    target_variables=configs.get('task.target_variables', required=True),
    forcing_variables=configs.get('task.forcing_variables', required=True),
    levels=configs.get('task.levels', required=True),
    input_duration=configs.get('task.input_duration', required=True))

datasource = ARCODataSource(dataset_path,
                            task=task_config,
                            target_lead_times=slice("1d", "60d"),
                            from_date=from_date,
                            to_date=None)

dataset = grain.MapDataset.source(datasource)
inputs, targets, forcings = dataset[0]
inputs, targets_template, forcings = xarray_jax.device_put((inputs, targets, forcings))

In [ ]:
# TODO: find out how tisr and progress variables are normalized in original GraphCast code

In [ ]:
mean_by_level, stddev_by_level, diffs_stddev_by_level, mask = xarray_jax.wrap_data((mean_by_level, stddev_by_level, diffs_stddev_by_level, mask), 
                                                                                   to_jax=True, np_contiguous=False)

@hk.without_apply_rng
@hk.transform
def predictor_fn(inputs, targets_template, forcings):
    predictor = trn_utils.get_predictor(configs=configs,
                                        mesh_data=mesh_data,
                                        grid_lat=grid_lat,
                                        grid_lon=grid_lon,
                                        grid_mask=grid_mask,
                                        mean_by_level=mean_by_level,
                                        stddev_by_level=stddev_by_level,
                                        mask_da=mask,
                                        diffs_stddev_by_level=diffs_stddev_by_level)
    return predictor(inputs=inputs, targets_template=targets_template, forcings=forcings)

rng = jax.random.key(configs.get('seed', required=True))
initial_params = predictor_fn.init(rng, inputs, targets_template.isel(time=0, drop=False), forcings.isel(time=0, drop=False))
predictor_fn_apply_jit = jax.jit(predictor_fn.apply)

In [ ]:
forecast = rollout.chunked_prediction(lambda rng, inputs, targets_template, forcings: predictor_fn_apply_jit(ckpt.params, 
                                                                                                             inputs, 
                                                                                                             targets_template, 
                                                                                                             forcings), 
                                             rng, inputs, targets_template, forcings) 

In [ ]:
get_dashboard(forecast, targets, projection=ccrs.Robinson())

In [ ]:
def rmse_fn(forecast, targets):
    squared_errs = (forecast - targets) ** 2
    mean_squared_errs = squared_errs.mean(dim=[dim for dim in targets.dims if dim != 'level' and dim != 'time'], skipna=True)
    return np.sqrt(mean_squared_errs)

In [ ]:
rmse = rmse_fn(forecast, targets)
rmse = rmse.assign_coords(time=np.datetime64(from_date) + rmse['time'])
rmse.hvplot(x='time')

In [ ]:
bias = (forecast - targets).mean(dim=[dim for dim in targets.dims if dim != 'level' and dim != 'time'], skipna=True)
bias = bias.assign_coords(time=np.datetime64(from_date) + bias['time'])
bias.hvplot(x='time')

In [ ]:
# def to_np(ds):
#     return xr.Dataset(data_vars={name: (data.dims, np.asarray(data.data.jax_array)) 
#                                  for name, data in ds.data_vars.items()}, 
#                       coords=ds.coords, 
#                       attrs=ds.attrs)

In [ ]:
# (loss, diagnostics), forecast = loss_and_predictions_jit(params=params, data=data, static_data=static_data)
# loss, diagnostics

In [ ]:
# %time
# (loss, diagnostics), forecast = loss_and_predictions_jit(params=params, data=data, static_data=static_data)

In [ ]:
# forecast = to_np(forecast)

In [ ]:
# (forecast - targets)['thetao'].isel(level=0, time=0, drop=True).hvplot.hist(bins=150, xlim=(-1, 1), normed='height')

In [ ]:
# increments = (targets.isel(time=0, drop=True) - inputs.isel(time=-1, drop=True))
# increments = increments.where(inputs['glorys_mask'], np.nan)
# normalized_increments = increments / diffs_stddev_by_level

In [ ]:
# forecast_normalized_increments = (forecast.isel(time=0, drop=True) - inputs.isel(time=0, drop=True)) / diffs_stddev_by_level
# forecast_normalized_increments = to_np(forecast_normalized_increments)

# get_dashboard(forecast_normalized_increments, title="# Forecast normalized increments", projection=ccrs.Robinson()) +\
# get_dashboard(forecast_normalized_increments - normalized_increments, title="# Forecast normalized increment errors", projection=ccrs.Robinson())

In [ ]:
# random_normalized_increments = (random_forecast.isel(time=0, drop=True) - inputs.isel(time=-1, drop=True)) / diffs_stddev_by_level
# random_normalized_increments = to_np(random_normalized_increments)

# get_dashboard(random_normalized_increments, title="# Random weights normalized increments", projection=ccrs.Robinson()) + \
# get_dashboard(random_normalized_increments - normalized_increments, title="# Random weights normalized increment errors", projection=ccrs.Robinson())